In [1]:
import copy
import math
import random
import time
from collections import OrderedDict, defaultdict
from typing import Union, List

import numpy as np
import torch
from matplotlib import pyplot as plt
from torch import nn
from torch.optim import *
from torch.optim.lr_scheduler import *
from torch.utils.data import DataLoader

try:
    from torchprofile import profile_macs
except ModuleNotFoundError:
    !pip -q install torchprofile
    from torchprofile import profile_macs


from torchvision.datasets import *
from torchvision.transforms import *
from tqdm.auto import tqdm
from torchvision import datasets, transforms



/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from utils.model_utils import get_model_macs, get_model_size, get_model_sparsity, get_num_parameters, get_sparsity
from utils.plot_utils import plot_num_parameters_distribution, plot_weight_distribution, plot_sensitivity_scan
from utils.prune_utils import fine_grained_prune, test_fine_grained_prune
from utils.sensitivity_analysis import sensitivity_scan
from utils.train_evaluate import train, evaluate

import torch
import torchvision.models as models


model = models.resnet18(num_classes=6)  # change num_classes if fine-tuned

model.load_state_dict(
    torch.load(
        "model/resnet18.pth", 
        map_location="cpu" if not torch.cuda.is_available() else None
    )
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

transform = Compose([
        Resize((150,150)),
        ToTensor(),
    ])
dataset = {}
for split in ["train", "test"]:
  dataset[split] = ImageFolder(
      root=f"/kaggle/input/intel-image-classification/seg_{split}/seg_{split}",transform=transform)
dataloader = {}
for split in ['train', 'test']:
  dataloader[split] = DataLoader(
    dataset[split],
    batch_size=16,
    shuffle=(split == 'train'),
    # num_workers=0,
    # pin_memory=True,
  )

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/intel-image-classification/seg_train/seg_train'